# Day11 — 브랜드 스타일 광고 카피 파인튜닝
> **Qwen2.5-VL-7B-Instruct + Unsloth FastVisionModel + SFT → DPO/SimPO**

## 설계 결정
| 항목 | 결정 | 이유 |
|------|------|------|
| 베이스 모델 | Qwen2.5-VL-7B-Instruct | 기획안 유지 |
| CPT 단계 | **생략** | 목적이 도메인 어휘 적응이 아닌 스타일 정렬 |
| 학습 순서 | SFT → DPO/SimPO | 스타일 정렬 후 선호 강화 |

## 파이프라인
```
Step0  환경 설정 + 모델 로드 + IMAGE_TOKEN_IDS 동적 추출
Step1  데이터 EDA (adcopy_sft.jsonl / adcopy_dpo.jsonl)
Step2  Baseline 측정 (학습 전 1회 — Claude-Judge)
Step3  SFT — 스타일 정렬 (합성 마스킹)
Step4  DPO/SimPO — chosen / rejected 선호 정렬
Step5  종합 평가 + 누적 비교표
Step6  (옵션) merge + HF 업로드
```

## 실험 변수 (S0 셀에서만 변경)
| 변수 | 실험 A | 실험 B |
|------|--------|--------|
| QLoRA rank | 8 | 16 |
| 학습 데이터 수 | 200쌍 | 350쌍 |

## SMART 목표
- AS-IS : Qwen2.5-VL base Claude-judge 스타일 일치도 40/100
- TO-BE : SFT 후 60/100 이상 (+20점)

## Step0 — 환경 설정 + 모델 로드

In [ ]:
# 최초 1회만 실행
!pip install unsloth[colab-new] -q
!pip install trl peft accelerate bitsandbytes anthropic -q
!pip install pillow nest_asyncio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/

In [ ]:
import os, json, random, re, time
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import nest_asyncio
nest_asyncio.apply()

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# 경로 설정
# Colab 사용 시 아래 주석 해제

BASE_DIR = Path('/content/drive/MyDrive/코리아아케데미_ai_포트폴리오')
# BASE_DIR   = Path('.')
IMAGE_ROOT = BASE_DIR / 'images'
SFT_JSONL  = BASE_DIR / 'adcopy_sft.jsonl'
DPO_JSONL  = BASE_DIR / 'adcopy_dpo.jsonl'

# ★ 실험 변수 (여기서만 변경)
LORA_RANK   = 8    # 실험 A: 8  /  실험 B: 16
N_SFT_TRAIN = 200  # 실험 A: 200  /  실험 B: 350

print(f'[CONFIG] model=Qwen2.5-VL-7B  LORA_RANK={LORA_RANK}  N_SFT_TRAIN={N_SFT_TRAIN}')

[CONFIG] model=Qwen2.5-VL-7B  LORA_RANK=8  N_SFT_TRAIN=200


In [ ]:
from unsloth import FastVisionModel

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

model, processor = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
print(f'[OK] {MODEL_ID}  dtype={model.dtype}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.90G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

[OK] Qwen/Qwen2.5-VL-7B-Instruct  dtype=torch.bfloat16


In [ ]:
# LoRA 부착 — vision encoder 동결, LLM proj 7개만 학습
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers    = False,
    finetune_language_layers  = True,
    finetune_attention_modules= True,
    finetune_mlp_modules      = True,
    r=LORA_RANK, lora_alpha=LORA_RANK*2,
    lora_dropout=0.05, bias='none', random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'[OK] LoRA rank={LORA_RANK}  trainable: {trainable/1e6:.1f}M / {total/1e9:.1f}B ({trainable/total*100:.3f}%)')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


[OK] LoRA rank=8  trainable: 20.2M / 5.0B (0.400%)


In [ ]:
# IMAGE_TOKEN_IDS 동적 추출 — 모든 collator 가 재사용
tok = processor.tokenizer
_names = ['<|vision_start|>', '<|vision_end|>', '<|image_pad|>']
IMAGE_TOKEN_IDS = [tok.convert_tokens_to_ids(t) for t in _names]
assert all(tid > 0 for tid in IMAGE_TOKEN_IDS)
for n, tid in zip(_names, IMAGE_TOKEN_IDS):
    print(f'  {n:<20} -> {tid}')

RESPONSE_TEMPLATE     = '<|im_start|>assistant\n'
RESPONSE_TEMPLATE_IDS = tok.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f'[OK] RESPONSE_TEMPLATE_IDS: {RESPONSE_TEMPLATE_IDS}')

  <|vision_start|>     -> 151652
  <|vision_end|>       -> 151653
  <|image_pad|>        -> 151655
[OK] RESPONSE_TEMPLATE_IDS: [151644, 77091, 198]


In [ ]:
# chat template 구조 점검 (SFT 마스킹 전 필수)
dummy = Image.new('RGB', (64, 64))
sample_msgs = [
    {'role': 'system',    'content': 'system'},
    {'role': 'user',      'content': [{'type':'image','image':dummy},
                                       {'type':'text', 'text':'MUJI 스타일로.'}]},
    {'role': 'assistant', 'content': '가볍고 부드러운 소재입니다.'},
]
sample_text = tok.apply_chat_template(sample_msgs, tokenize=False, add_generation_prompt=False)
print(sample_text[:400])
print()
assert '<|vision_start|>' in sample_text
assert RESPONSE_TEMPLATE  in sample_text
print('[OK] chat template 구조 정상')

<|im_start|>system
system<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>MUJI 스타일로.<|im_end|>
<|im_start|>assistant
가볍고 부드러운 소재입니다.<|im_end|>


[OK] chat template 구조 정상


## Step1 — 데이터 EDA

In [ ]:
from collections import Counter

sft_raw = [json.loads(l) for l in open(SFT_JSONL, encoding='utf-8')]
dpo_raw = [json.loads(l) for l in open(DPO_JSONL, encoding='utf-8')]

print(f'SFT: {len(sft_raw)}건  |  DPO: {len(dpo_raw)}건')
print(f'스타일: {dict(Counter(r.get("style_id") for r in sft_raw))}')
print(f'브랜드: {dict(Counter(r.get("brand")    for r in sft_raw))}')

lengths = [len(r['output']) for r in sft_raw]
print(f'카피 길이 — 평균 {np.mean(lengths):.1f}자 / max {max(lengths)} / min {min(lengths)}')

print('\n[SFT 샘플 각 2건]')
for sid in ['A','B']:
    pool = [r for r in sft_raw if r.get('style_id')==sid]
    for r in random.sample(pool, min(2,len(pool))):
        print(f'  [{sid}] {r["output"]}')

if dpo_raw:
    r = dpo_raw[0]
    print(f'\n[DPO 샘플]')
    print(f'  chosen:   {r["chosen"]}')
    print(f'  rejected: {r["rejected"]}')

SFT: 95건  |  DPO: 99건
스타일: {'A': 68, 'B': 27}
브랜드: {'muji': 50, 'uniqlo': 45}
카피 길이 — 평균 33.0자 / max 50 / min 22

[SFT 샘플 각 2건]
  [A] 편안한 밴딩 허리와 자연스러운 플레어 실루엣으로 일상에 편안함을 더합니다.
  [A] 필요한 것만 담을 수 있는 자연스러운 면 소재 가방입니다.
  [B] 매일 입고 싶은 부드러운 베이지, 어떤 스타일에도 자연스럽게 어울립니다.
  [B] 매일 입고 싶은 편안함, 세이지그린 폴로가 일상을 가볍게 만듭니다.

[DPO 샘플]
  chosen:   시간이 지나도 변하지 않는 라운드 프레임의 기본입니다.
  rejected: 시간이 흘러도 변하지 않는 클래식의 가치. MUJI 라운드 안경은 불필요한 장식 없이 본질만 담았습니다. 당신의 일상에 자연스럽게 스며드는 베이직한 아름다움을 경험해보세요. #무지안경 #클래식라운드 #베이직스타일 #심플라이프


In [ ]:
# product_id 단위 train / hold-out 분리
all_pids = list(set(r['product_id'] for r in sft_raw))
random.shuffle(all_pids)
n_hold       = max(int(len(all_pids)*0.15), 10)
holdout_pids = set(all_pids[:n_hold])

sft_holdout = [r for r in sft_raw if r['product_id'] in holdout_pids]
pool_all    = [r for r in sft_raw if r['product_id'] not in holdout_pids]

# Style A / B 균형 유지
pa = [r for r in pool_all if r.get('style_id')=='A']
pb = [r for r in pool_all if r.get('style_id')=='B']
n  = N_SFT_TRAIN // 2
sft_train = pa[:n] + pb[:n]
random.shuffle(sft_train)

print(f'Train: {len(sft_train)}건 (A:{n} B:{n})  Hold-out: {len(sft_holdout)}건')

with open(BASE_DIR/'adcopy_holdout.jsonl','w',encoding='utf-8') as f:
    for r in sft_holdout: f.write(json.dumps(r,ensure_ascii=False)+'\n')
print('[OK] adcopy_holdout.jsonl 저장')

Train: 81건 (A:100 B:100)  Hold-out: 14건
[OK] adcopy_holdout.jsonl 저장


## Step2 — Baseline 측정 (학습 전 1회)

In [ ]:
import anthropic
if not os.environ.get('ANTHROPIC_API_KEY'):
    from getpass import getpass
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')
claude = anthropic.Anthropic()

def generate_copy(m, proc, image, instruction, max_new_tokens=80):
    msgs = [{'role':'user','content':[{'type':'image','image':image},
                                       {'type':'text','text':instruction}]}]
    text = proc.tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inputs = proc(text=[text],images=[image],return_tensors='pt').to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    return proc.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

STYLE_TONE = {
    'A': 'MUJI형: 소재 기능 객관 서술, 문어체(~입니다), 수식어 최소, 50자 이내',
    'B': 'UNIQLO형: 라이프스타일 감성 제안, 구어체(~이겁니다), 35자 이내',
}

def claude_judge(product_id, category, style_id, copy):
    prompt = (
        f'아래 광고 카피를 3축 rubric 으로 채점하세요. JSON 만 응답.\n'
        f'제품: {product_id} ({category})\n'
        f'스타일: Style {style_id} — {STYLE_TONE.get(style_id,"")}\n'
        f'카피: {copy}\n\n'
        '{"style_fit":<1~5>,"copy_quality":<1~5>,"product_match":<1~5>,"reason":"<20자"}'
    )
    resp = claude.messages.create(
        model='claude-sonnet-4-20250514', max_tokens=200,
        messages=[{'role':'user','content':prompt}]
    )
    raw = resp.content[0].text
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        return json.loads(m.group())
    return {'style_fit':1,'copy_quality':1,'product_match':1,'reason':'parse_error'}

def avg100(scores, key):
    v = [s[key] for s in scores if key in s]
    return round(float(np.mean(v))*20,1) if v else 0.0

def run_eval(m, proc, label, sample):
    scores = []
    print(f'[{label}] 평가 {len(sample)}건')
    for i, rec in enumerate(sample):
        img_path = IMAGE_ROOT / rec['image'].replace('images/','')
        try: pil = Image.open(img_path).convert('RGB')
        except: continue
        pred  = generate_copy(m, proc, pil, rec['instruction'])
        score = claude_judge(rec.get('product_id',''), rec.get('category',''),
                              rec.get('style_id','A'), pred)
        scores.append(score)
        print(f'  [{i+1}][{rec["style_id"]}] {pred[:30]}  fit={score["style_fit"]} '
              f'qual={score["copy_quality"]} match={score["product_match"]}')
        time.sleep(0.5)
    sf,cq,pm = avg100(scores,'style_fit'), avg100(scores,'copy_quality'), avg100(scores,'product_match')
    row = {'단계':label,'style_fit':sf,'copy_quality':cq,'product_match':pm,'종합':round((sf+cq+pm)/3,1)}
    print(f'  -> 종합 {row["종합"]}/100\n')
    return row

print('[OK] 유틸 정의 완료')

ANTHROPIC_API_KEY: ··········
[OK] 유틸 정의 완료


In [ ]:
FastVisionModel.for_inference(model)

EVAL_N      = min(20, len(sft_holdout))
eval_sample = random.sample(sft_holdout, EVAL_N)
results_log = []

row = run_eval(model, processor, 'Baseline (Qwen2.5-VL raw)', eval_sample)
results_log.append(row)
print(f'목표 60/100 달성까지: +{max(0, 60-row["종합"]):.1f}점 필요')

[Baseline (Qwen2.5-VL raw)] 평가 14건


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/tmp/ipykernel_5440/301980165.py:29: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  resp = claude.messages.create(


  [1][A] "순수한 디자인과 깨끗한 라이프스타일을 선사하는 MUJ  fit=2 qual=4 match=3


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [2][B] "더 깔끔하고 세련된 UNIQLO 스타일, 당신의 일상  fit=2 qual=4 match=3


...

  [12][A] "더 깔끔하고, 더 깔끔한 스타일, MUJI의 신선한   fit=2 qual=3 match=4


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [13][A] "순수한 아름다움을 담은, MUJI의 기본 티셔츠."  fit=2 qual=3 match=4


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [14][A] "미니멀리즘과 깔끔함을 담은 MUJI 스타일의 캐주얼한  fit=2 qual=3 match=4
  -> 종합 56.7/100

목표 60/100 달성까지: +3.3점 필요


## Step3 — SFT (스타일 정렬)

In [ ]:
def find_subsequence(seq, subseq):
    n, m = len(seq), len(subseq)
    for i in range(n-m+1):
        if seq[i:i+m]==subseq: return i
    return -1

def sft_collate_fn(examples):
    """
    3단 합성 마스킹:
      1) 이미지 토큰 -> -100
      2) RESPONSE_TEMPLATE 이전 -> -100
      3) padding -> -100
    학습 영역 = assistant 응답(카피 1문장)만
    """
    texts, images = [], []
    for ex in examples:
        img_path = IMAGE_ROOT / ex['image'].replace('images/','')
        try:    img = Image.open(img_path).convert('RGB')
        except: img = Image.new('RGB',(224,224),(200,200,200))

        msgs = [
            {'role':'system','content':(
                '당신은 한국 패션 쇼핑몰 전문 카피라이터입니다. '
                '제품 사진을 보고 지정된 브랜드 스타일에 맞는 '
                '한국어 광고 카피를 한 문장으로 작성합니다.')},
            {'role':'user','content':[
                {'type':'image','image':img},
                {'type':'text', 'text':ex['instruction']}]},
            {'role':'assistant','content':ex['output']},
        ]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text); images.append(img)

    batch = processor(text=texts, images=images, return_tensors='pt',
                      padding=True, truncation=True, max_length=512)
    labels = batch['input_ids'].clone()

    for tid in IMAGE_TOKEN_IDS:                            # 1) 이미지 토큰
        labels[labels==tid] = -100
    for i in range(labels.size(0)):                        # 2) RESPONSE_TEMPLATE 이전
        pos = find_subsequence(batch['input_ids'][i].tolist(), RESPONSE_TEMPLATE_IDS)
        if pos < 0: labels[i,:] = -100
        else:       labels[i,:pos+len(RESPONSE_TEMPLATE_IDS)] = -100
    labels[batch['attention_mask']==0] = -100              # 3) padding

    batch['labels'] = labels
    return batch

print('[OK] sft_collate_fn 정의')

[OK] sft_collate_fn 정의


In [ ]:
# ★ 마스킹 검증 — 학습 전 필수
tb  = sft_collate_fn([sft_train[0]])
inp = tb['input_ids'][0].tolist()
lbl = tb['labels'][0].tolist()
img_in_labels = sum(1 for t in lbl if t in IMAGE_TOKEN_IDS and t!=-100)
learnable     = sum(1 for t in lbl if t!=-100)
print(f'이미지 토큰 in labels: {img_in_labels}  (★ 0 이어야 정상)')
print(f'학습 가능 토큰 수    : {learnable}       (= 카피 길이)')
assert img_in_labels==0, '이미지 토큰 마스킹 실패'
assert learnable>0,      '학습 가능 토큰 없음'
print('[OK] 마스킹 검증 통과')

ValueError: Mismatch in `image` token count between text and `input_ids`. Got ids=[444] and text=[1156]. Likely due to `truncation='max_length'`. Please disable truncation or increase `max_length`.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from torch.utils.data import Dataset

class ListDataset(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self):    return len(self.d)
    def __getitem__(self,i): return self.d[i]

FastVisionModel.for_training(model)

sft_trainer = SFTTrainer(
    model=model,
    args=TrainingArguments(
        output_dir='sft_adapter',
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        save_strategy='epoch',
        dataloader_num_workers=0,
        seed=SEED,
        report_to='none',
    ),
    train_dataset=ListDataset(sft_train),
    data_collator=sft_collate_fn,
    tokenizer=tok,
    dataset_text_field=None,
    packing=False,
)

print(f'[OK] SFT Trainer  데이터={len(sft_train)}건  rank={LORA_RANK}  lr=2e-5  1epoch')

t0 = time.time()
res = sft_trainer.train()
print(f'[OK] SFT 완료 ({(time.time()-t0)/60:.1f}분)  loss={res.training_loss:.4f}')

sft_trainer.model.save_pretrained('sft_adapter/best')
tok.save_pretrained('sft_adapter/best')
print('[OK] sft_adapter/best 저장')

In [ ]:
FastVisionModel.for_inference(model)
row = run_eval(model, processor, f'SFT (rank={LORA_RANK}, n={N_SFT_TRAIN})', eval_sample)
results_log.append(row)

gain = row['종합'] - results_log[0]['종합']
print(f'baseline -> SFT: {gain:+.1f}점  |  목표: {"✅ 달성" if row["종합"]>=60 else "❌ 미달"}')

if row['종합'] < 45:
    print('[FALLBACK] 45점 미만 -> S0 셀에서 LORA_RANK=16, N_SFT_TRAIN=350 으로 변경 후 재실행')

## Step4 — DPO/SimPO (선호 정렬)

In [ ]:
# SFT 어댑터 merge -> DPO 새 LoRA 시작점
print('[INFO] merge_and_unload ...')
merged = sft_trainer.model.merge_and_unload()
merged.save_pretrained('sft_merged'); tok.save_pretrained('sft_merged')
print('[OK] sft_merged 저장')

dpo_model = FastVisionModel.get_peft_model(
    merged,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_RANK, lora_alpha=LORA_RANK*2,
    lora_dropout=0.05, bias='none', random_state=SEED,
)
FastVisionModel.for_training(dpo_model)
print('[OK] DPO 용 새 LoRA 부착')

In [ ]:
def make_dpo_record(rec):
    img_path = IMAGE_ROOT / rec['image'].replace('images/','')
    try:    img = Image.open(img_path).convert('RGB')
    except: return None
    sid = rec.get('style_id','A')
    name = 'MUJI형' if sid=='A' else 'UNIQLO형'
    return {
        'prompt':[
            {'role':'system','content':f'Style {sid}({name}) 카피라이터입니다.'},
            {'role':'user','content':[
                {'type':'image','image':img},
                {'type':'text', 'text':rec.get('prompt', rec.get('instruction',''))},
            ]},
        ],
        'chosen':  [{'role':'assistant','content':rec['chosen']}],
        'rejected':[{'role':'assistant','content':rec['rejected']}],
    }

dpo_dataset = [d for d in (make_dpo_record(r) for r in dpo_raw) if d]
print(f'[OK] DPO dataset: {len(dpo_dataset)}건')
if dpo_dataset:
    print(f'  chosen:   {dpo_dataset[0]["chosen"][0]["content"]}')
    print(f'  rejected: {dpo_dataset[0]["rejected"][0]["content"]}')

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_done = False
if len(dpo_dataset) < 10:
    print(f'[SKIP] DPO 데이터 {len(dpo_dataset)}건 < 10건 — Day10 에서 페어 추가 생성 후 재실행')
else:
    dpo_trainer = DPOTrainer(
        model=dpo_model,
        ref_model=None,
        args=DPOConfig(
            output_dir='dpo_adapter',
            loss_type='simpo',
            beta=2.0, simpo_gamma=1.4,
            learning_rate=5e-7,
            num_train_epochs=1,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            lr_scheduler_type='cosine', warmup_ratio=0.1,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=5, save_strategy='epoch',
            seed=SEED, report_to='none',
        ),
        train_dataset=ListDataset(dpo_dataset),
        processing_class=tok,
    )
    print(f'[OK] DPO(SimPO)  {len(dpo_dataset)}건  beta=2.0  gamma=1.4  lr=5e-7')

    t0 = time.time()
    dpo_res = dpo_trainer.train()
    print(f'[OK] DPO 완료 ({(time.time()-t0)/60:.1f}분)  loss={dpo_res.training_loss:.4f}')
    dpo_trainer.model.save_pretrained('dpo_adapter/best'); tok.save_pretrained('dpo_adapter/best')
    print('[OK] dpo_adapter/best 저장')
    dpo_done = True

## Step5 — 종합 평가 + 누적 비교표

In [ ]:
if dpo_done:
    FastVisionModel.for_inference(dpo_model)
    row = run_eval(dpo_model, processor, f'DPO/SimPO (rank={LORA_RANK})', eval_sample)
    results_log.append(row)
    final_model = dpo_model
else:
    final_model = model
    print('[INFO] DPO 생략 — SFT 모델 최종 사용')

In [ ]:
import pandas as pd

df = pd.DataFrame(results_log).set_index('단계')
print('='*60)
print('★ 누적 비교표 (100점 스케일)')
print('='*60)
print(df.to_string())
print('='*60)

final = results_log[-1]['종합']
print(f'\n최종 {final}/100  |  {"✅ 목표 달성" if final>=60 else "❌ 미달 — 실험 B 권장"}')

df.to_csv(BASE_DIR/'cumulative_comparison.csv', encoding='utf-8-sig')
print('[OK] cumulative_comparison.csv 저장')

In [ ]:
# 추론 데모 — hold-out 1장 x 2스타일
FastVisionModel.for_inference(final_model)
rec = random.choice(sft_holdout)
img = Image.open(IMAGE_ROOT / rec['image'].replace('images/','')).convert('RGB')

print(f'[데모] {rec["product_id"]}\n')
for sid, instr in [
    ('A','이 제품 사진을 보고 MUJI 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.'),
    ('B','이 제품 사진을 보고 UNIQLO 스타일의 한국어 광고 카피를 한 문장으로 작성하세요.'),
]:
    pred = generate_copy(final_model, processor, img, instr)
    print(f'  Style {sid}: {pred}')

## Step6 — (옵션) merge + HF 업로드

In [ ]:
HF_REPO = 'your-username/adcopy-qwen25vl-7b'  # 변경 필요
UPLOAD  = False

if UPLOAD:
    final_merged = final_model.merge_and_unload()
    from huggingface_hub import login
    login()
    final_merged.push_to_hub(HF_REPO)
    processor.tokenizer.push_to_hub(HF_REPO)
    print(f'[OK] https://huggingface.co/{HF_REPO}')
else:
    print('[SKIP] UPLOAD=True 로 변경 후 실행')

---
## 정리

| 산출물 | 다음 단계 |
|--------|----------|
| `sft_adapter/best` | SFT 어댑터 |
| `sft_merged/` | DPO base / Gradio 서빙 |
| `dpo_adapter/best` | DPO 어댑터 |
| `cumulative_comparison.csv` | 발표 슬라이드 |
| `adcopy_holdout.jsonl` | 평가 전용 |

### 실험 A/B 전환
```python
# S0 셀에서 아래 값만 변경 후 전체 재실행
LORA_RANK   = 16  # 8 -> 16
N_SFT_TRAIN = 350 # 200 -> 350
```

### Day13 서빙 연결
```python
from unsloth import FastVisionModel
model, processor = FastVisionModel.from_pretrained('sft_merged', load_in_4bit=True)
FastVisionModel.for_inference(model)
```